# TP2 — 02 : Traitements PySpark et indicateurs

**Binôme : Enora et Edwin** · Projet BigData Commerce Analytics

Ce notebook lit les commandes et paiements créés par le notebook 01, étudie les transformations, les actions, les partitions et Spark UI, puis prépare les indicateurs pour le jour 3.

Exécuter d'abord le notebook 01 et garder MongoDB démarré. Les montants correspondent à une simulation ; les dates de paiement retiennent la plus ancienne des dates synthétiques de chaque panier.

## 1. Démarrer Spark en local

    Le connecteur MongoDB 11.1 est compatible avec Spark 4. Le mode `local[*]` utilise les cœurs disponibles sur cet ordinateur.

In [1]:
import os
import sys
import time
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

java_tp = Path(sys.prefix) / "jdk-17"
if java_tp.exists() and not os.environ.get("JAVA_HOME"):
    os.environ["JAVA_HOME"] = str(java_tp)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("TP2 BigData Commerce Analytics")
    .config("spark.jars.ivy", os.path.join(sys.prefix, ".ivy2"))
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:11.1.0")
    .config("spark.mongodb.read.connection.uri", "mongodb://127.0.0.1:27017")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Version Spark :", spark.version)
print("Mode :", spark.sparkContext.master)
print("Spark UI :", spark.sparkContext.uiWebUrl)

:: loading settings :: url = jar:file:/home/faton/Documents/FATON/10%20-%20Cours/B2/03%20-%20Bloc%20Specialisation/E33%20-%20Big%20Data/TP/BigData_Commerce_Analytics_DummyJSON/B2_ENORA_EDWIN_BigData/jour-2/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/faton/Documents/FATON/10 - Cours/B2/03 - Bloc Specialisation/E33 - Big Data/TP/BigData_Commerce_Analytics_DummyJSON/B2_ENORA_EDWIN_BigData/jour-2/venv/.ivy2/cache
The jars for the packages stored in: /home/faton/Documents/FATON/10 - Cours/B2/03 - Bloc Specialisation/E33 - Big Data/TP/BigData_Commerce_Analytics_DummyJSON/B2_ENORA_EDWIN_BigData/jour-2/venv/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-28f133b5-aec9-4a52-8185-f5b4a8700289;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;11.1.0 in central


	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
:: resolution report :: resolve 1559ms :: artifacts dl 5ms
	:: modules in use:
	org.mongodb#bson;5.1.4 from central in [default]
	org.mongodb#bson-record-codec;5.1.4 from central in [default]
	org.mongodb#mongodb-driver-core;5.1.4 from central in [default]
	org.mongodb#mongodb-driver-sync;5.1.4 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.13;11.1.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   5   |   1   |   0   |   0   

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Version Spark : 4.0.1
Mode : local[*]
Spark UI : http://pop-os:4040


## 2. Lire les collections MongoDB

    Le champ `_id` est l'identifiant technique MongoDB. Il sert à identifier un document ; ce n'est pas le numéro métier de commande ou de panier.

In [2]:
commandes = (
    spark.read.format("mongodb")
    .option("database", "commerce")
    .option("collection", "raw_commandes")
    .load()
)
paiements = (
    spark.read.format("mongodb")
    .option("database", "commerce")
    .option("collection", "raw_payments")
    .load()
)

print("Schéma des commandes :")
commandes.printSchema()
print("Schéma des paiements :")
paiements.printSchema()

Schéma des commandes :
root
 |-- _id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cart_discounted_total: double (nullable = true)
 |-- cart_id: integer (nullable = true)
 |-- cart_total: double (nullable = true)
 |-- category: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- discounted_line_total: double (nullable = true)
 |-- line_total: double (nullable = true)
 |-- numero_ligne: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)

Schéma des paiements :
root
 |-- _id: string (nullable = true)
 |-- cart_id: integer (nullable = true)
 |-- date_paiement: string (nullable = true)
 |-- montant: double (nullable = true)
 |-- order_id

In [3]:
commandes.select("cart_id", "user_id", "product_id", "category", "quantity", "discounted_line_total").show(5, truncate=False)
paiements.select("cart_id", "order_id", "user_id", "montant", "date_paiement", "statut").show(5, truncate=False)

nombre_commandes = commandes.count()
nombre_paniers = commandes.select("cart_id").distinct().count()
nombre_paiements = paiements.count()
nombre_paiements_uniques = paiements.select("cart_id").distinct().count()
print("Lignes de commandes :", nombre_commandes)
print("Paniers distincts :", nombre_paniers)
print("Paiements :", nombre_paiements)
print("Paniers distincts dans les paiements :", nombre_paiements_uniques)
assert nombre_commandes == 800
assert nombre_paniers == nombre_paiements == nombre_paiements_uniques == 208

+-------+-------+----------+------------------+--------+---------------------+
|cart_id|user_id|product_id|category          |quantity|discounted_line_total|
+-------+-------+----------+------------------+--------+---------------------+
|1      |1      |162       |tops              |4       |105.41               |
|1      |1      |113       |motorcycle        |3       |10547.97             |
|1      |1      |122       |smartphones       |3       |839.76               |
|1      |1      |138       |sports-accessories|2       |17.67                |
|2      |2      |86        |mens-shirts       |5       |93.12                |
+-------+-------+----------+------------------+--------+---------------------+
only showing top 5 rows
+-------+----------+-------+--------+-------------+-------+
|cart_id|order_id  |user_id|montant |date_paiement|statut |
+-------+----------+-------+--------+-------------+-------+
|1      |CART-00001|1      |11510.81|2025-03-02   |PAID   |
|2      |CART-00002|2    

Lignes de commandes : 800
Paniers distincts : 208
Paiements : 208
Paniers distincts dans les paiements : 208


## 3. Transformations, actions et lazy evaluation

    `select`, `filter`, `withColumn`, `groupBy` et `orderBy` décrivent des transformations. Elles construisent un plan sans déclencher à elles seules le calcul complet. Une action, par exemple `show`, `count`, `first` ou `collect`, demande un résultat et déclenche les calculs nécessaires.

    Nous comptons les jobs visibles dans Spark UI avant et après la construction d'un plan, puis après une action. Les autres cellules ont déjà lancé des jobs ; nous comparons donc les nombres et non une page nécessairement vide.

In [4]:
port_ui = urlparse(spark.sparkContext.uiWebUrl).port
url_jobs = f"http://127.0.0.1:{port_ui}/api/v1/applications/{spark.sparkContext.applicationId}/jobs"

def nombre_jobs_ui():
    reponse = requests.get(url_jobs, timeout=10)
    reponse.raise_for_status()
    return len(reponse.json())

jobs_avant = nombre_jobs_ui()

calcul_en_attente = (
    commandes.select("category", "quantity", "discounted_line_total")
    .filter(F.col("quantity") > 1)
    .withColumn("prix_unitaire_remise", F.round(F.col("discounted_line_total") / F.col("quantity"), 2))
    .groupBy("category")
    .agg(F.round(F.sum("discounted_line_total"), 2).alias("montant_lignes"))
    .orderBy(F.desc("montant_lignes"))
)

jobs_apres_transformations = nombre_jobs_ui()
print("Jobs avant les transformations :", jobs_avant)
print("Jobs après les transformations :", jobs_apres_transformations)
calcul_en_attente.show(5, truncate=False)
time.sleep(0.5)
jobs_apres_action = nombre_jobs_ui()
print("Jobs après l'action show() :", jobs_apres_action)
assert jobs_avant == jobs_apres_transformations
assert jobs_apres_action > jobs_apres_transformations

Jobs avant les transformations : 12
Jobs après les transformations : 12


+--------------+--------------+
|category      |montant_lignes|
+--------------+--------------+
|vehicle       |1595455.28    |
|motorcycle    |594708.43     |
|mens-watches  |544863.9      |
|womens-watches|194369.13     |
|laptops       |82138.1       |
+--------------+--------------+
only showing top 5 rows


Jobs après l'action show() : 14


**Observation :** la préparation du plan n'a ajouté aucun job ; l'action `show()` a lancé l'exécution. Spark peut optimiser le plan avant de l'exécuter. Selon l'action et les échanges de données, une action peut produire un ou plusieurs jobs.

## 4. Partitions et Spark UI

    Une partition est un morceau de données. Une **Task** traite une partition pour une étape ; un **Stage** regroupe des Tasks qui peuvent être exécutées dans la même étape ; un **Job** regroupe les Stages nécessaires à un calcul. Un échange de données (*shuffle*) peut séparer plusieurs Stages.

    Nous affichons d'abord les partitions d'origine, puis nous répartissons les 800 lignes en quatre partitions pour observer leur contenu.

In [5]:
print("Partitions de lecture MongoDB :", commandes.rdd.getNumPartitions())
commandes_reparties = commandes.repartition(4)
print("Partitions après repartition(4) :", commandes_reparties.rdd.getNumPartitions())
repartition_lignes = commandes_reparties.rdd.mapPartitions(lambda lignes: [sum(1 for _ in lignes)]).collect()
for numero, nombre in enumerate(repartition_lignes):
    print("Partition", numero, ":", nombre, "lignes")
assert sum(repartition_lignes) == nombre_commandes

Partitions de lecture MongoDB : 1


Partitions après repartition(4) : 4


Partition 0 : 200 lignes
Partition 1 : 200 lignes
Partition 2 : 200 lignes
Partition 3 : 200 lignes


Dans Spark UI, l'onglet **Jobs** liste les calculs déclenchés ; **Stages** montre leurs étapes et leurs Tasks ; **SQL / DataFrame** présente les plans des requêtes ; **Executors** affiche les ressources et **Environment** les paramètres. Les captures du dossier `captures/` ont été prises pendant l'exécution de ce notebook. Les durées en local sur 800 lignes ne représentent pas les performances d'un cluster.

## 5. Indicateurs pour le jour 3

    Les montants de panier sont répétés sur plusieurs lignes : on ne les additionne jamais dans le DataFrame des lignes. Nous distinguons le **montant commandé** par catégorie de l'**encaissé**, qui ne conserve que les paniers dont le paiement est `PAID`.

In [6]:
commandes_par_categorie = commandes.groupBy("category").agg(
    F.round(F.sum("discounted_line_total"), 2).alias("ca_commande")
)
paniers_payes = paiements.filter(F.col("statut") == "PAID").select("cart_id")
lignes_payees = commandes.join(paniers_payes, on="cart_id", how="inner")
encaisse_par_categorie = lignes_payees.groupBy("category").agg(
    F.round(F.sum("discounted_line_total"), 2).alias("ca_encaisse")
)
ca_categories = (
    commandes_par_categorie.join(encaisse_par_categorie, on="category", how="left")
    .fillna({"ca_encaisse": 0.0})
    .orderBy(F.desc("ca_commande"), F.asc("category"))
)
print("Montants commandés et encaissés par catégorie :")
ca_categories.show(50, truncate=False)

Montants commandés et encaissés par catégorie :


+-------------------+-----------+-----------+
|category           |ca_commande|ca_encaisse|
+-------------------+-----------+-----------+
|vehicle            |1682064.05 |1374964.95 |
|motorcycle         |609460.61  |404546.47  |
|mens-watches       |593063.86  |446483.7   |
|womens-watches     |249189.51  |216806.68  |
|laptops            |85006.24   |51469.78   |
|furniture          |77674.97   |49322.28   |
|smartphones        |73130.95   |62735.15   |
|mobile-accessories |26160.99   |21354.64   |
|tablets            |14890.16   |9098.16    |
|mens-shoes         |8278.9     |6321.99    |
|womens-bags        |5849.91    |2217.36    |
|kitchen-accessories|5714.33    |3837.02    |
|fragrances         |4958.05    |4316.35    |
|womens-dresses     |4064.13    |3418.41    |
|sports-accessories |3776.68    |2871.56    |
|home-decoration    |2747.3     |2031.52    |
|tops               |2233.24    |1725.16    |
|groceries          |1881.5     |1306.71    |
|womens-shoes       |1538.66    |9

In [7]:
top_clients = (
    paiements.filter(F.col("statut") == "PAID")
    .groupBy("user_id")
    .agg(F.round(F.sum("montant"), 2).alias("ca_encaisse"))
    .orderBy(F.desc("ca_encaisse"), F.asc("user_id"))
    .limit(10)
)
print("Top 10 des clients selon les paiements encaissés :")
top_clients.show(10, truncate=False)

Top 10 des clients selon les paiements encaissés :
+-------+-----------+
|user_id|ca_encaisse|
+-------+-----------+
|30     |155507.16  |
|52     |130175.28  |
|95     |129783.31  |
|83     |125679.0   |
|33     |125461.58  |
|18     |124978.81  |
|45     |121385.95  |
|155    |110543.89  |
|78     |109890.08  |
|27     |100496.36  |
+-------+-----------+



In [8]:
nombre_refuses = paiements.filter(F.col("statut") == "REFUSED").count()
taux_refus = round(nombre_refuses / nombre_paiements * 100, 2)
print("Paiements refusés :", nombre_refuses, "/", nombre_paiements)
print("Taux de refus :", taux_refus, "%")

indicateur_refus = spark.createDataFrame(
    [(nombre_refuses, nombre_paiements, taux_refus)],
    ["nombre_refuses", "nombre_tentatives", "taux_refus_pourcent"],
)

Paiements refusés : 37 / 208
Taux de refus : 17.79 %


In [9]:
evolution_mensuelle = (
    paiements.filter(F.col("statut") == "PAID")
    .withColumn("mois", F.date_format(F.to_date("date_paiement"), "yyyy-MM"))
    .groupBy("mois")
    .agg(F.round(F.sum("montant"), 2).alias("ca_encaisse"))
    .orderBy("mois")
)
print("Évolution mensuelle du montant encaissé :")
evolution_mensuelle.show(30, truncate=False)

Évolution mensuelle du montant encaissé :


+-------+-----------+
|mois   |ca_encaisse|
+-------+-----------+
|2025-01|977810.05  |
|2025-02|245583.44  |
|2025-03|527737.26  |
|2025-04|146639.35  |
|2025-05|231045.7   |
|2025-06|200283.04  |
|2025-07|28521.47   |
|2025-08|27910.65   |
|2025-09|189195.28  |
|2025-10|94509.37   |
+-------+-----------+



### Contrôler les indicateurs et les conserver

    La somme de l'encaissé par catégorie et celle par mois doivent chacune correspondre au total des paiements `PAID`. Les quatre fichiers CSV seront réutilisables pour les visualisations du jour 3.

In [10]:
total_paye = paiements.filter(F.col("statut") == "PAID").agg(F.round(F.sum("montant"), 2)).first()[0]
total_categories = ca_categories.agg(F.round(F.sum("ca_encaisse"), 2)).first()[0]
total_mois = evolution_mensuelle.agg(F.round(F.sum("ca_encaisse"), 2)).first()[0]
print("Total des paiements PAID :", total_paye)
print("Total encaissé par catégorie :", total_categories)
print("Total encaissé par mois :", total_mois)
assert abs(total_paye - total_categories) < 0.02
assert abs(total_paye - total_mois) < 0.02

dossier_resultats = Path("resultats")
dossier_resultats.mkdir(exist_ok=True)
exports = {
    "ca_par_categorie.csv": ca_categories,
    "top_10_clients.csv": top_clients,
    "taux_refus.csv": indicateur_refus,
    "evolution_mensuelle.csv": evolution_mensuelle,
}
for nom, dataframe in exports.items():
    lignes = [ligne.asDict() for ligne in dataframe.collect()]
    pd.DataFrame(lignes).to_csv(dossier_resultats / nom, index=False)
    print("Fichier créé :", dossier_resultats / nom)

Total des paiements PAID : 2669235.61
Total encaissé par catégorie : 2669235.61
Total encaissé par mois : 2669235.61


Fichier créé : resultats/ca_par_categorie.csv
Fichier créé : resultats/top_10_clients.csv


Fichier créé : resultats/taux_refus.csv
Fichier créé : resultats/evolution_mensuelle.csv


In [11]:
premiere_categorie = ca_categories.first()
premier_client = top_clients.first()
meilleur_mois = evolution_mensuelle.orderBy(F.desc("ca_encaisse"), F.asc("mois")).first()
print("Catégorie avec le plus grand montant commandé :", premiere_categorie["category"], "-", premiere_categorie["ca_commande"])
print("Client avec le plus grand montant encaissé :", premier_client["user_id"], "-", premier_client["ca_encaisse"])
print("Part des paiements refusés :", taux_refus, "%")
print("Mois simulé avec le plus grand montant encaissé :", meilleur_mois["mois"], "-", meilleur_mois["ca_encaisse"])
print("Les dates de paiement sont synthétiques : l'évolution mensuelle n'est pas un historique réel.")

Catégorie avec le plus grand montant commandé : vehicle - 1682064.05
Client avec le plus grand montant encaissé : 30 - 155507.16
Part des paiements refusés : 17.79 %
Mois simulé avec le plus grand montant encaissé : 2025-01 - 977810.05
Les dates de paiement sont synthétiques : l'évolution mensuelle n'est pas un historique réel.


## 6. Réponses de synthèse

    - **Collection MongoDB / DataFrame Spark :** une collection stocke des documents dans la base ; un DataFrame représente des données structurées que Spark peut transformer et analyser.
    - **Transformation / action :** une transformation prépare un nouveau plan, par exemple `filter` ; une action demande un résultat, par exemple `count`.
    - **Lazy evaluation :** Spark attend une action pour exécuter les transformations nécessaires et peut optimiser le plan avant le calcul.
    - **Partitions / Tasks / Stages / Jobs :** les données sont découpées en partitions ; des Tasks les traitent dans des Stages, qui composent un Job.
    - **Chiffre d'affaires encaissé :** seul `PAID` correspond à un paiement accepté ; `REFUSED` et `PENDING` ne constituent pas un encaissement.

    Les données clients de DummyJSON sont des données de démonstration. Dans un projet réel, il faudrait limiter les champs personnels collectés et protéger l'accès à la base.

### Observation d’un Stage dans Spark UI

Nous relevons les Tasks d’un Stage terminé contenant au moins quatre Tasks lorsque cela est possible. La capture `spark_ui_stage_tasks.png` montre ce même Stage.

In [12]:
url_stages = f"http://127.0.0.1:{port_ui}/api/v1/applications/{spark.sparkContext.applicationId}/stages"
stages = requests.get(url_stages, timeout=10).json()
stages_termines = [stage for stage in stages if stage.get("status") == "COMPLETE"]
stages_multi = [stage for stage in stages_termines if stage.get("numTasks", 0) >= 4]
stage_observe = max(stages_multi or stages_termines, key=lambda stage: stage["stageId"])
print("Stage observé :", stage_observe["stageId"])
print("Tasks terminées :", stage_observe["numCompleteTasks"], "/", stage_observe["numTasks"])
print("Tasks échouées :", stage_observe["numFailedTasks"])

Stage observé : 103
Tasks terminées : 16 / 16
Tasks échouées : 0


### Observations SQL / DataFrame, Executors et Environment

Nous interrogeons les informations de la session visibles dans ces onglets pour relever les exécutions, les ressources et la configuration réellement utilisées. Les captures correspondantes sont dans le dossier `captures/`.

In [13]:
url_application = f"http://127.0.0.1:{port_ui}/api/v1/applications/{spark.sparkContext.applicationId}"

reponse_sql = requests.get(url_application + "/sql", params={"length": 1000}, timeout=10)
reponse_sql.raise_for_status()
executions_sql = reponse_sql.json()
print("SQL / DataFrame :", len(executions_sql), "exécutions de requêtes visibles")

reponse_executors = requests.get(url_application + "/executors", timeout=10)
reponse_executors.raise_for_status()
for executor in reponse_executors.json():
    print("Executor :", executor["id"], "- cœurs :", executor["totalCores"], "- Tasks terminées :", executor["completedTasks"])

reponse_environment = requests.get(url_application + "/environment", timeout=10)
reponse_environment.raise_for_status()
environnement = reponse_environment.json()
proprietes_spark = dict(environnement["sparkProperties"])
print("Environment - mode :", proprietes_spark.get("spark.master"))
print("Environment - partitions de shuffle :", proprietes_spark.get("spark.sql.shuffle.partitions"))
print("Environment - version Java :", environnement["runtime"]["javaVersion"])

SQL / DataFrame : 21 exécutions de requêtes visibles
Executor : driver - cœurs : 16 - Tasks terminées : 91
Environment - mode : local[*]
Environment - partitions de shuffle : 4
Environment - version Java : 17.0.21.1 (Eclipse Adoptium)


In [14]:
print("Spark UI pendant cette session :", spark.sparkContext.uiWebUrl)
print("Consulter Jobs, Stages, SQL / DataFrame, Executors et Environment.")

Spark UI pendant cette session : http://pop-os:4040
Consulter Jobs, Stages, SQL / DataFrame, Executors et Environment.
